# Final Image Model Training

**ISIC 2024 – Skin Cancer Detection with 3D-TBP**

This notebook trains the final image classification model using the complete ISIC 2024 training dataset.

Unlike the cross-validation notebook, no validation split is created. The architecture and hyperparameters have already been selected, so the objective is to obtain the final model used during inference.

## Why train a final model?

Cross-validation is used to estimate model performance and select the training configuration. After the best architecture and hyperparameters have been identified, a new model is trained using the complete training dataset in order to maximize the amount of information available during learning.

The resulting model is the one used to generate predictions for the competition test set.

## Configuration

Define the dataset paths, selected backbone architecture, and training hyperparameters used to train the final model.

In [1]:
# ==========================================================
# Model
# ==========================================================

MODEL_NAME = "resnet18"
# MODEL_NAME = "efficientnet_b0"

# ==========================
# Paths
# ==========================

HDF5_PATH = "/kaggle/input/competitions/isic-2024-challenge/train-image.hdf5"
META_PATH = "/kaggle/input/competitions/isic-2024-challenge/train-metadata.csv"

# ==========================
# Training
# ==========================

N_FOLDS = 5
EPOCHS = 5

BATCH_SIZE = 64

LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

# ==========================
# DataLoader
# ==========================

NUM_WORKERS = 2
PIN_MEMORY = True

# ==========================
# Image
# ==========================

IMAGE_SIZE = 224

# ==========================
# Seed
# ==========================

SEED = 42

## Imports

Import the required libraries together with the project modules responsible for dataset loading, model definition, and training.

In [2]:
import torch
import torch.nn as nn
import random
import numpy as np
import pandas as pd
import sys

from torchvision import transforms
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from pathlib import Path

In [3]:
OUTPUT_DIR = Path("/kaggle/working")

MODEL_DIR = OUTPUT_DIR / "models"
HISTORY_DIR = OUTPUT_DIR / "history"
OOF_DIR = OUTPUT_DIR / "oof"

MODEL_DIR.mkdir(exist_ok=True)
HISTORY_DIR.mkdir(exist_ok=True)
OOF_DIR.mkdir(exist_ok=True)

In [4]:
sys.path.append(
    "/kaggle/input/datasets/wagneraugustoaff/isic2024-code/src/image"
)

from image_model import ISICModelRsn
from image_model import ISICModelEff
from image_dataset import ISICDataset
from image_utils import seed_everything
from image_train import train_one_epoch

In [5]:
seed_everything(SEED)

In [6]:
# Device
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

## Dataset Preparation

Load the complete ISIC 2024 training metadata and create the image dataset used to train the final model.

In [7]:
df = pd.read_csv(
    META_PATH,
    low_memory=False
)

## Data Augmentation

Define the image transformations applied during training. Since the final model is trained using all available samples, only the training augmentation pipeline is required.

In [8]:
train_transform = transforms.Compose([

    transforms.Resize((224,224)),

    transforms.RandomHorizontalFlip(p=0.5),

    transforms.RandomRotation(20),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.1,
        hue=0.02,
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),

])

## Training

Train the final image classification model using the complete training dataset.

In [9]:
train_df = df.reset_index(drop=True)

num_pos = (train_df["target"] == 1).sum()
num_neg = (train_df["target"] == 0).sum()

# Compute the positive-class weight to compensate for class imbalance.
pos_weight = torch.tensor(
    [num_neg / num_pos],
    dtype=torch.float32,
    device=device,
)

criterion = torch.nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)

# Apply data augmentation during training.
train_dataset = ISICDataset(
    dataframe=train_df,
    image_path=HDF5_PATH,
    transform=train_transform,
)


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

if MODEL_NAME == "resnet18":
    model = ISICModelRsn().to(device)

elif MODEL_NAME == "efficientnet_b0":
    model = ISICModelEff().to(device)

else:
    raise ValueError(f"Unknown model: {MODEL_NAME}")

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

for epoch in range(EPOCHS):

    train_loss = train_one_epoch(
        model=model,
        train_loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
    )

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Loss: {train_loss:.5f}"
    )

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 157MB/s] 


KeyboardInterrupt: 

## Save Final Model

Save the trained model weights for use during inference and Kaggle submission generation.

In [ ]:
torch.save(
    model.state_dict(),
    f"/kaggle/working/{MODEL_NAME}_final.pth",
)

## Summary

The resulting model is the final image classifier used during inference. Its prediction probabilities are combined with the CatBoost metadata model through the Logistic Regression stacking model to generate the final competition submission.